# AcademiQ - Notebook 04: Student Factor Analysis

This notebook analyzes the student-related factors associated with academic performance using:
1. **Supervised Univariate Feature Selection** via `SelectKBest` with the $F$-regression score (`f_regression`).
2. **Trained Multiple Linear Regression (MLR) Coefficient Analysis** to examine directional relationships across features.

> **Important Methodological Note & Limitation:**
> Statistical association and feature importance scores measure predictive correlation with `Exam_Score`. They **do not prove causality**. An observed relationship between higher study hours or attendance and higher examination scores reflects predictive association within this observational cohort, not a guarantee that manipulating one factor in isolation will produce a deterministic score gain.

In [ ]:
import os
from pathlib import Path

def get_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "data" / "raw").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current

PROJECT_ROOT = get_project_root()
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression

print(f"Project root resolved to: {PROJECT_ROOT}")
print("Libraries imported successfully.")

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
train_file = processed_dir / "AcademiQ_Train_Cleaned.csv"
test_file = processed_dir / "AcademiQ_Test_Cleaned.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

target = "Exam_Score"
X_train = train_df.drop(columns=[target]).reset_index(drop=True)
X_test = test_df.drop(columns=[target]).reset_index(drop=True)
y_train = train_df[target].reset_index(drop=True)
y_test = test_df[target].reset_index(drop=True)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")
train_df.head()

In [ ]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Preprocessor for feature selection: StandardScaler on numeric, OneHotEncoder on categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    sparse_threshold=0.0
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(f"Processed training matrix: {X_train_processed.shape}")
print(f"Total expanded features:   {len(feature_names)}")

In [ ]:
# Fit SelectKBest using univariate f_regression
k = min(10, X_train_processed.shape[1])

selector = SelectKBest(
    score_func=f_regression,
    k=k
)

X_train_selected = selector.fit_transform(
    X_train_processed,
    y_train
)

X_test_selected = selector.transform(
    X_test_processed
)

selected_features = feature_names[selector.get_support()]

print("Top 10 Selected Features:")
for idx, feat in enumerate(selected_features, 1):
    print(f"  {idx:2d}. {feat}")

In [ ]:
feature_scores = pd.DataFrame({
    "Feature": feature_names,
    "F_Score": selector.scores_,
    "P_Value": selector.pvalues_
})

feature_scores = feature_scores.sort_values(
    by="F_Score",
    ascending=False
).reset_index(drop=True)

print("Top 15 Features Ranked by F-Score:")
display(feature_scores.head(15))

In [ ]:
# Visualization of Top 10 Features based on F-Score
top_features = feature_scores.head(10)

plt.figure(figsize=(10, 5))
plt.barh(
    top_features["Feature"],
    top_features["F_Score"],
    color="#2b5c8f"
)
plt.title("Top 10 Student Factors by F-Score (SelectKBest)", fontsize=13, fontweight="bold")
plt.xlabel("F-Score", fontsize=11)
plt.ylabel("Student Feature / Attribute", fontsize=11)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Examination of Trained Multiple Linear Regression Pipeline Coefficients
mlr_model_path = PROJECT_ROOT / "models" / "AcademiQ_Multiple_Linear_Regression.pkl"

if mlr_model_path.exists():
    mlr_pipeline = joblib.load(mlr_model_path)
    regressor = mlr_pipeline.named_steps["regressor"]
    preprocessor_mlr = mlr_pipeline.named_steps["preprocessor"]
    
    mlr_features = preprocessor_mlr.get_feature_names_out()
    mlr_coefs = regressor.coef_
    
    coef_df = pd.DataFrame({
        "Feature": mlr_features,
        "Coefficient": mlr_coefs,
        "Abs_Impact": np.abs(mlr_coefs)
    }).sort_values(by="Abs_Impact", ascending=False).reset_index(drop=True)
    
    print("Trained MLR Model Top Coefficients (Standardized Scale):")
    display(coef_df.head(10))
    print(f"MLR Intercept: {regressor.intercept_:.4f}")
else:
    print("MLR model not found at models/AcademiQ_Multiple_Linear_Regression.pkl")

In [ ]:
# Save feature selection artifacts
results_fa_dir = PROJECT_ROOT / "results" / "feature_analysis"
results_fa_dir.mkdir(parents=True, exist_ok=True)
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

feature_scores.to_csv(results_fa_dir / "AcademiQ_Feature_Selection_Results.csv", index=False)
sel_df = pd.DataFrame({"Selected_Feature": selected_features})
sel_df.to_csv(results_fa_dir / "AcademiQ_Selected_Features.csv", index=False)

# Preserve existing verified feature selector artifact; only dump if not already present
selector_path = models_dir / "AcademiQ_Feature_Selector.pkl"
if not selector_path.exists():
    joblib.dump(selector, selector_path)

print("Feature selection results successfully saved/verified:")
print(f"  - {results_fa_dir / 'AcademiQ_Feature_Selection_Results.csv'}")
print(f"  - {results_fa_dir / 'AcademiQ_Selected_Features.csv'}")
print(f"  - {selector_path}")


## Summary & Key Findings

1. **Dominant Predictors:** `Attendance` ($F = 2666.48$) and `Hours_Studied` ($F = 1244.44$) show the strongest statistical association with `Exam_Score`, followed by `Previous_Scores` ($F = 162.90$) and `Tutoring_Sessions` ($F = 131.86$).
2. **Environmental & Contextual Factors:** High levels of `Access_to_Resources` ($F = 119.58$) and `Parental_Involvement` ($F = 107.67$) also show meaningful positive associations.
3. **Alignment with Regression:** These findings directly support our Multiple Linear Regression model, explaining why including all multi-dimensional factors lifts $R^2$ from $0.23$ (study hours alone) to $0.77$ (all factors combined).